# Tarefa 2 — Baseline do fluxo e rotulagem OK / RISCO / FALHA

Lê `data/raw/ripe_mesh_ping.csv` (Tarefa 1) e `data/raw/metadata_coleta.json`. **Não treina árvore.**

Ordem obrigatória: 1 inspeção do bruto → 2 ficha de baseline (só Período A) → 3 métricas relativas → 4 rótulo na ordem da tabela → 5 recorte temporal dentro de B.

Execute de cima para baixo, a partir da raiz do repositório. Saídas: `data/interim/`, `data/processed/`.

## Conexão dos Notebooks


In [ ]:
import sys
from pathlib import Path

# Detecta se está no Google Colab ou em ambiente local
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_PROJETO = Path("/content/drive/MyDrive/Projeto interdisciplinar")
except (ImportError, ModuleNotFoundError):
    RAIZ_PROJETO = Path(__file__).resolve().parents[2] if "__file__" in locals() else Path.cwd().parents[1]

# Configura o caminho absoluto de PROC
PROC = RAIZ_PROJETO / "data" / "processed"

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/Projeto interdisciplinar")
except (ImportError, ModuleNotFoundError):
    BASE = Path("../../").resolve()

RAW = BASE / "data" / "raw"
INTERIM = BASE / "data" / "interim"
PROC = BASE / "data" / "processed"

for p in (INTERIM, PROC):
    p.mkdir(parents=True, exist_ok=True)

# --- parâmetros da Tarefa 2 (valores do enunciado) ---
PISO_RTT_VALIDOS = 1500
JANELA = 5
FALHA_PERDA, FALHA_N5_TIMEOUT, FALHA_Z, FALHA_N5_AUM80 = 10, 3, 3.5, 2
AUM80 = 80
RISCO_Z_MIN, RISCO_AUM_MIN, RISCO_AUM_MAX, RISCO_JIT, RISCO_N5 = 2.0, 30, 80, 3.0, 2
PROPORCAO = (0.50, 0.20, 0.30)

meta = json.loads((RAW / "metadata_coleta.json").read_text(encoding="utf-8"))
CORTE_AB, FIM_B = meta["corte_a_b_epoch"], meta["fim_b_epoch"]
print("Período A:", meta["periodo_a_utc"])
print("Período B:", meta["periodo_b_utc"])

## 1. Inspeção do bruto (ainda sem rótulo)

In [ ]:
cat_cols = ["fluxo_id", "dst_addr", "origem_pais", "destino_pais", "rota_id"]
df = pd.read_csv(RAW / "ripe_mesh_ping.csv", dtype={c: str for c in cat_cols},
                 usecols=lambda c: c not in ("rtts", "dst_name"))

df["periodo"] = np.where(df.timestamp < CORTE_AB, "A", np.where(df.timestamp < FIM_B, "B", "fora"))

aud = {
    "N_bruto": int(len(df)),
    "N_fluxos": int(df.fluxo_id.nunique()),
    "duplicatas_fluxo_timestamp": int(df.duplicated(["fluxo_id", "timestamp"]).sum()),
    "rtt_vazio": int(df.latencia_ms.isna().sum()),
    "rtt_vazio_pct": round(float(df.latencia_ms.isna().mean() * 100), 2),
    "rtt_gravado_como_zero_ou_negativo": int((df.latencia_ms <= 0).sum()),
    "avg_zero_ou_negativo": int((df.avg <= 0).sum()),
    "timeout_inconsistente": int(((df.latencia_ms.isna()) & (df.timeout_atual == 0)).sum()),
    "linhas_fora_de_A_e_B": int((df.periodo == "fora").sum()),
}
a_max = df[df.periodo == "A"].groupby("fluxo_id").timestamp.max()
b_min = df[df.periodo == "B"].groupby("fluxo_id").timestamp.min()
aud["fluxos_com_timestamp_em_A_e_B"] = int((a_max.reindex(b_min.index) >= b_min).sum())
aud["linhas_A"], aud["linhas_B"] = int((df.periodo == "A").sum()), int((df.periodo == "B").sum())
print(json.dumps(aud, indent=2, ensure_ascii=False))

assert aud["rtt_gravado_como_zero_ou_negativo"] == 0, "RTT ausente gravado como 0/-1!"
assert aud["avg_zero_ou_negativo"] == 0
assert aud["fluxos_com_timestamp_em_A_e_B"] == 0, "A e B compartilham timestamp"
df = df[df.periodo != "fora"].copy()

In [ ]:
# Fluxos esperados no plano x fluxos que apareceram (a Tarefa 1 achou 121 de 123)
plano_p = RAW / "plano_coleta.json"
if plano_p.exists():
    plano = json.loads(plano_p.read_text(encoding="utf-8"))
    esperados = {f"{s['probe_id']}|{a['dst_addr']}" for a in plano["alvos"] for s in a["origens"]}
    presentes = set(df.fluxo_id)
    print("Esperados:", len(esperados), "| presentes:", len(presentes))
    print("Planejados SEM nenhuma linha:", sorted(esperados - presentes))
    print("Presentes fora do plano:", sorted(presentes - esperados))

# Cobertura por fluxo no Período A: onde está o problema dos fluxos abaixo do piso?
A = df[df.periodo == "A"]
cob = (A.groupby(["fluxo_id", "rota_id", "probe_id", "dst_addr"])
         .agg(n_medicoes_A=("timestamp", "size"), n_rtt_validos_A=("latencia_ms", "count"))
         .reset_index())
cob["pct_vazio_A"] = (100 * (1 - cob.n_rtt_validos_A / cob.n_medicoes_A)).round(1)
abaixo = cob[cob.n_rtt_validos_A < PISO_RTT_VALIDOS].sort_values(["dst_addr", "probe_id"])
print(f"\nFluxos abaixo de {PISO_RTT_VALIDOS} RTT válidos no A: {len(abaixo)} de {len(cob)}")
display(abaixo)

In [ ]:
# Padrão do problema: o PROBE falha em todos os alvos, ou o ALVO falha para todos os probes?
cob["abaixo"] = cob.n_rtt_validos_A < PISO_RTT_VALIDOS
print("Por probe (fração de fluxos abaixo do piso):")
print(cob.groupby("probe_id").abaixo.agg(["sum", "size"]).query("sum > 0"))
print("\nPor alvo:")
print(cob.groupby("dst_addr").abaixo.agg(["sum", "size"]).query("sum > 0"))

In [ ]:
# Tabela de trabalho (a auditoria vai para o diário)
df.to_csv(INTERIM / "medicoes_com_periodo.csv.gz", index=False, compression="gzip")
(INTERIM / "auditoria_bruto.json").write_text(json.dumps(aud, indent=2), encoding="utf-8")

## 2. Ficha de baseline (só Período A, uma linha por `fluxo_id`)

`MAD` = mediana de |RTT − mediana| (sem o fator 1,4826; ele entra na fórmula do `z_robusto`).
Fluxo com menos de 1.500 RTT válidos no A → `baseline_insuficiente`: **sem mediana, sem classe**.
O Período B não entra em nenhuma conta desta seção.

In [ ]:
linhas = []
for fid, g in df[df.periodo == "A"].groupby("fluxo_id"):
    v = g.latencia_ms.dropna()
    f = {"fluxo_id": fid, "rota_id": g.rota_id.iloc[0], "origem_pais": g.origem_pais.iloc[0],
         "destino_pais": g.destino_pais.iloc[0], "n_medicoes_A": len(g), "n_rtt_validos_A": len(v),
         "prop_resposta": len(v) / len(g)}
    if len(v) >= PISO_RTT_VALIDOS:
        med = v.median()
        f.update(status="ok", mediana=med, MAD=(v - med).abs().median(),
                 iqr=v.quantile(0.75) - v.quantile(0.25),
                 jitter_tipico=g.jitter_ms.dropna().median() if g.jitter_ms.notna().any() else np.nan,
                 perda_tipica=g.perda_pct.median())
    else:
        f.update(status="baseline_insuficiente")
    linhas.append(f)

ficha = pd.DataFrame(linhas)
# fluxos que só existem em B (sem nenhuma linha no A) também não têm ficha
so_b = sorted(set(df.fluxo_id) - set(ficha.fluxo_id))
if so_b:
    extra = df[df.fluxo_id.isin(so_b)].groupby("fluxo_id").agg(
        rota_id=("rota_id", "first"), origem_pais=("origem_pais", "first"),
        destino_pais=("destino_pais", "first")).reset_index()
    extra["status"] = "baseline_insuficiente"; extra["n_medicoes_A"] = 0; extra["n_rtt_validos_A"] = 0
    ficha = pd.concat([ficha, extra], ignore_index=True)

cols = ["fluxo_id", "rota_id", "origem_pais", "destino_pais", "status", "n_medicoes_A", "n_rtt_validos_A",
        "mediana", "MAD", "iqr", "jitter_tipico", "perda_tipica", "prop_resposta"]
ficha = ficha[cols].sort_values("fluxo_id")
ficha.to_csv(PROC / "baseline_por_fluxo.csv", index=False)     # congelada daqui em diante

ok = ficha[ficha.status == "ok"]
print("Fluxos com ficha:", len(ok), "| excluídos (baseline_insuficiente):", (ficha.status != "ok").sum())
print("Fluxos com MAD = 0:", int((ok.MAD == 0).sum()), "| com jitter_tipico = 0:", int((ok.jitter_tipico == 0).sum()))

In [ ]:
# Lista dos excluídos e o motivo
excl = ficha[ficha.status != "ok"][["fluxo_id", "rota_id", "n_medicoes_A", "n_rtt_validos_A", "prop_resposta"]]
excl = excl.assign(motivo=f"RTT válidos no A < {PISO_RTT_VALIDOS}")
display(excl)

# Exemplo auditável: um fluxo curto e um longo (medianas muito diferentes, as duas são "normal")
curto = ok[ok.rota_id.isin(["DE-DE", "AR-AR", "BR-BR"])].sort_values("mediana").head(1)
longo = ok[ok.rota_id.isin(["BR-JP", "BR-IN", "BR-ZA"])].sort_values("mediana").tail(1)
display(pd.concat([curto, longo]))

## 3. Métricas de cada medição do Período B

Calculadas só com a ficha congelada de cada `fluxo_id`.

**Decisão a registrar no diário:** `n5_*` olha as últimas 5 medições *do mesmo fluxo* (linhas, não minutos), incluindo a atual,
na série contínua A+B. Assim, as 4 primeiras linhas de B usam o final de A (guia §7: "a métrica em t usa só o mesmo fluxo com instante ≤ t").
Nenhuma linha de A é rotulada.

In [ ]:
# Mudança aqui (para passar todos os campos da baseline)
fi = ficha[ficha.status == "ok"].set_index("fluxo_id")
w = df[df.fluxo_id.isin(fi.index)].sort_values(["fluxo_id", "timestamp"]).copy()
for c in ["mediana", "MAD", "iqr", "jitter_tipico", "perda_tipica", "prop_resposta"]:
    w[c] = w.fluxo_id.map(fi[c]) # adiciona os campos perda_tipíca, prop_resposta 

num = w.latencia_ms - w.mediana                       # NaN se RTT ausente (nunca 0 ms)
den = np.where(w.MAD > 0, 1.4826 * w.MAD, np.maximum(w.iqr / 1.349, 1.0))   # MAD=0 -> max(IQR/1,349; 1 ms)
w["z_robusto"] = num / den                            # MAD=0 e RTT=mediana -> 0
w["aumento_pct"] = num / w.mediana * 100

ja, jt = w.jitter_ms, w.jitter_tipico
with np.errstate(divide="ignore", invalid="ignore"):
    jr = ja / jt
jr = np.where((jt == 0) & (ja == 0), 1.0, jr)         # tipico 0 e atual 0 -> 1
jr = np.where((jt == 0) & (ja > 0), RISCO_JIT, jr)    # tipico 0 e atual > 0 -> >= 3 (usamos 3,0)
w["jitter_relativo"] = jr                             # NaN: jitter atual vazio -> critério não dispara

w["timeout_atual"] = (w.latencia_ms.isna() | (w.perda_pct >= 100)).astype(int)

crit_risco = (((w.z_robusto >= RISCO_Z_MIN) & (w.z_robusto < FALHA_Z))
              | ((w.aumento_pct >= RISCO_AUM_MIN) & (w.aumento_pct <= RISCO_AUM_MAX))
              | (w.jitter_relativo >= RISCO_JIT))
w["crit_risco_atual"] = crit_risco.astype(int)
w["_aum80"] = (w.aumento_pct > AUM80).astype(int)

def n5(col):
    return w.groupby("fluxo_id")[col].transform(lambda s: s.rolling(JANELA, min_periods=1).sum()).astype(int)

w["n5_timeout"] = n5("timeout_atual")
w["n5_aumento80"] = n5("_aum80")
w["n5_risco"] = n5("crit_risco_atual")

assert (w.timeout_atual.values == df.loc[w.index, "timeout_atual"].values).all(), "timeout recalculado difere do bruto"
w[["z_robusto", "aumento_pct", "jitter_relativo", "n5_timeout", "n5_aumento80", "n5_risco"]].describe().T

## 4. Rotulagem — parar na primeira linha verdadeira (só Período B)

FALHA ganha de RISCO; RISCO ganha de OK. `regra_disparada` guarda a linha da tabela que decidiu (1 a 6) para auditar.

In [ ]:
conds = [
    w.perda_pct >= FALHA_PERDA,                       # 1 FALHA
    w.n5_timeout >= FALHA_N5_TIMEOUT,                 # 2 FALHA
    w.z_robusto >= FALHA_Z,                           # 3 FALHA (NaN = sem RTT -> não dispara)
    w.n5_aumento80 >= FALHA_N5_AUM80,                 # 4 FALHA
    (w.crit_risco_atual == 1) & (w.n5_risco >= RISCO_N5),   # 5 RISCO
]
w["regra_disparada"] = np.select(conds, [1, 2, 3, 4, 5], default=6)
w["classe"] = np.select(conds, ["FALHA"] * 4 + ["RISCO"], default="OK")

B = w[w.periodo == "B"].copy()              # o Período A NÃO é rotulado
print("Contagem no Período B:")
print(B.classe.value_counts().reindex(["OK", "RISCO", "FALHA"]).fillna(0).astype(int))
print((B.classe.value_counts(normalize=True).reindex(["OK", "RISCO", "FALHA"]) * 100).round(2).astype(str) + " %")
print("\nPor regra que disparou:\n", B.regra_disparada.value_counts().sort_index())

In [ ]:
# Independência da rota: a classe não pode depender de RTT absoluto nem do nome da rota
print("OK com RTT > 100 ms (caminho longo saudável):", int(((B.classe == "OK") & (B.latencia_ms > 100)).sum()))
print("Não-OK com RTT < 100 ms (caminho curto degradado):", int(((B.classe != "OK") & (B.latencia_ms < 100)).sum()))
print()
print((pd.crosstab(B.rota_id, B.classe, normalize="index") * 100).round(1).reindex(columns=["OK", "RISCO", "FALHA"]))

In [ ]:
# Três exemplos auditáveis (linhas reais)
show = ["fluxo_id", "rota_id", "timestamp_utc", "latencia_ms", "mediana", "z_robusto", "aumento_pct",
        "jitter_relativo", "perda_pct", "n5_timeout", "n5_aumento80", "n5_risco", "classe", "regra_disparada"]
longas = B.rota_id.isin(["BR-JP", "BR-IN", "BR-ZA"])
curtas = B.rota_id.isin(["DE-DE", "AR-AR", "BR-BR"])
ex_ok = B[(B.classe == "OK") & longas & (B.latencia_ms > 200) & (B.z_robusto.abs() < 1)].head(1)
ex_risco = B[B.classe == "RISCO"].head(1)
ex_falha = B[(B.classe == "FALHA") & (curtas | (B.regra_disparada.isin([1, 2])))].head(1)
exemplos = pd.concat([ex_ok, ex_risco, ex_falha])[show]
display(exemplos.T)

## 5. Recorte temporal dentro de B (ainda sem treinar)

Por fluxo, em ordem de tempo: treino = trecho mais antigo, validação = meio, teste = mais recente (50/20/30 de partida).
O recorte vem **depois** do rótulo e das janelas `n5_*`, que usam só o passado. Nenhuma linha de A entra.

In [ ]:
B = B.sort_values(["fluxo_id", "timestamp"]).copy()
pos = B.groupby("fluxo_id").cumcount()
tam = B.groupby("fluxo_id").timestamp.transform("size")
frac = pos / tam
B["bloco"] = np.where(frac < PROPORCAO[0], "treino",
              np.where(frac < PROPORCAO[0] + PROPORCAO[1], "validacao", "teste"))
assert (B.periodo == "B").all()

tab = pd.crosstab(B.bloco, B.classe).reindex(index=["treino", "validacao", "teste"],
                                              columns=["OK", "RISCO", "FALHA"]).fillna(0).astype(int)
tab["N"] = tab.sum(axis=1)
datas = B.groupby("bloco").timestamp_utc.agg(["min", "max"]).reindex(["treino", "validacao", "teste"])
print(tab.join(datas))

sem = [(b, c) for b in tab.index for c in ("RISCO", "FALHA") if tab.loc[b, c] == 0]
print("\nBlocos sem alguma classe:", sem if sem else "nenhum")
if sem:
    print("-> ajuste PROPORCAO (ex.: 0.45, 0.20, 0.35) e rode esta seção de novo; registre a nova proporção.")

In [ ]:
FEATURES = ["z_robusto", "aumento_pct", "jitter_relativo", "perda_pct", "timeout_atual",
            "n5_timeout", "n5_aumento80", "n5_risco"]
PROIBIDAS = ["fluxo_id", "probe_id", "dst_addr", "rota_id", "origem_pais", "destino_pais",
             "latencia_ms", "avg", "min", "max", "mediana", "MAD", "iqr", "jitter_ms", "jitter_tipico",
             "msm_id", "timestamp", "timestamp_utc", "regra_disparada", "bloco", "periodo"]
assert not set(FEATURES) & set(PROIBIDAS)

# 1. Salva a tabela com dados completos de auditoria para o diário da Tarefa 2
auditoria_cols = ["timestamp", "timestamp_utc", "fluxo_id", "probe_id", "dst_addr", "rota_id",
                  "origem_pais", "destino_pais", "latencia_ms", "jitter_ms"]
saida = B[auditoria_cols + FEATURES + ["classe", "regra_disparada", "bloco"]]
saida.to_csv(PROC / "dataset_B_rotulado.csv", index=False)

# 2. Salva o DATASET FINAL EXATO com as colunas que a professora pediu:
COLUNAS_BASELINE = ["mediana", "MAD", "jitter_tipico", "perda_tipica", "prop_resposta"]
FEATURES_RELATIVAS = ["z_robusto", "aumento_pct", "jitter_relativo", "timeout_atual",
                      "n5_timeout", "n5_aumento80", "n5_risco"]

dataset_final = B.rename(columns={
    "jitter_ms": "jitter",
    "latencia_ms": "latencia",
    "perda_pct": "perda_pacotes",
    "classe": "status"
})[
    ["timestamp", "jitter", "latencia", "perda_pacotes", "status"]
    + COLUNAS_BASELINE
    + FEATURES_RELATIVAS
    + ["fluxo_id", "bloco"]
]
dataset_final.to_csv(PROC / "dataset_final_treino.csv", index=False)

recorte = {
    "proporcao_inicial": PROPORCAO,
    "N": {b: int(tab.loc[b, "N"]) for b in tab.index},
    "datas_utc": {b: [datas.loc[b, "min"], datas.loc[b, "max"]] for b in datas.index},
    "classes_por_bloco": tab.drop(columns="N").to_dict("index"),
    "criterio": "por fluxo, em ordem de timestamp, dentro do Período B",
}
(PROC / "recorte_temporal.json").write_text(json.dumps(recorte, indent=2, ensure_ascii=False), encoding="utf-8")

aud.update({
    "fluxos_com_ficha": int((ficha.status == "ok").sum()),
    "fluxos_excluidos": int((ficha.status != "ok").sum()),
    "contagem_classes_B": B.classe.value_counts().to_dict(),
    "recorte": recorte,
})
(PROC / "auditoria_tarefa2.json").write_text(json.dumps(aud, indent=2, ensure_ascii=False), encoding="utf-8")
exemplos.T.to_csv(PROC / "exemplos_auditaveis.csv")

print("Tudo salvo com sucesso em:", PROC)
print("\nColunas do dataset_final_treino.csv gerado:")
print(list(dataset_final.columns))

In [ ]:
dic = f'''# Dicionário de dados v0.2

## Bruto (`data/raw/ripe_mesh_ping.csv`, da Tarefa 1)
`timestamp` (epoch UTC), `timestamp_utc`, `msm_id`, `probe_id`, `dst_addr`, `dst_name`, `fluxo_id` (= probe_id|dst_addr),
`origem_pais`, `destino_pais`, `rota_id`, `sent`, `rcvd`, `avg`, `min`, `max`, `latencia_ms` (= avg; vazio se sem resposta),
`perda_pct`, `jitter_ms` (desvio-padrão amostral de result[].rtt; vazio com menos de 2 respostas), `timeout_atual`, `rtts`.
RTT ausente fica vazio: nunca 0, nunca -1.

## Ficha (`baseline_por_fluxo.csv`, só Período A, uma linha por fluxo)
`mediana`, `MAD` (mediana de |RTT - mediana|, sem fator 1,4826), `iqr` (usado só se MAD = 0), `jitter_tipico`, `perda_tipica`,
`prop_resposta`, `status` (`ok` ou `baseline_insuficiente`: menos de {PISO_RTT_VALIDOS} RTT válidos no A; sem estatísticas e sem classe).

## Métricas (Período B, com a ficha congelada)
{chr(10).join(f"- `{c}`" for c in FEATURES)}

`z_robusto`/`aumento_pct` ficam vazios sem RTT; `jitter_relativo` fica vazio sem jitter atual. `n5_*` = últimas {JANELA} medições do fluxo
(linhas), incluindo a atual, na série contínua A+B (as 4 primeiras de B usam o fim de A). Jitter típico 0 e atual > 0 vale 3,0.

## Classe
`classe` ∈ {{OK, RISCO, FALHA}} pela tabela da Tarefa 2, na ordem, parando na primeira linha verdadeira; `regra_disparada` (1 a 6) registra qual.
Só o Período B é rotulado.

## Colunas proibidas como atributo da árvore
{", ".join(f"`{c}`" for c in PROIBIDAS)}
(país, IP, rota e fluxo identificam o caminho; RTT absoluto não pode substituir as métricas relativas.)
Atributos permitidos: {", ".join(f"`{c}`" for c in FEATURES)}.

## Recorte (`bloco`)
`treino` / `validacao` / `teste`: por fluxo, em ordem de tempo, dentro de B. Proporção inicial {PROPORCAO}. Detalhes em `recorte_temporal.json`.
'''
(PROC / "dicionario_v0.2.md").write_text(dic, encoding="utf-8")
print(dic[:600])